In [1]:
import os
import pathlib

import pandas as pd
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from tqdm.auto import tqdm

root_dir, in_notebook = init_notebook()
profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
def count_parquets(row):
    extracted_features_dir = (
        profile_base_dir
        / "data"
        / row["patient"]
        / "extracted_features"
        / row["well_fov"]
    )
    if extracted_features_dir.exists():
        return sum(1 for p in extracted_features_dir.glob("*.parquet"))

In [3]:
patients_file_path = pathlib.Path(f"{root_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
patients = pd.read_csv(
    patients_file_path,
    header=None,
    names=["patient_id"],
).patient_id.tolist()
# patients = ["NF0014_T1"]  # --- IGNORE ---
load_file_path = pathlib.Path(
    f"{root_dir}/4.processing_image_based_profiles/load_data/load_file.txt"
).resolve()
load_file_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
rows = []

for patient in tqdm(patients, desc="Patients"):
    extracted_features_dir = profile_base_dir / "data" / patient / "extracted_features"
    if not extracted_features_dir.exists():
        print(f"No extracted_features directory for patient {patient}; skipping.")
        continue

    well_fovs = sorted(
        d.name
        for d in extracted_features_dir.iterdir()
        if d.is_dir() and "run_stats" not in d.name
    )
    for well_fov in well_fovs:
        rows.append({"patient": patient, "well_fov": well_fov})

df = pd.DataFrame(rows)
print(f"Total patient/well_fov combinations: {df.shape[0]}")

Patients:   0%|          | 0/13 [00:00<?, ?it/s]

Total patient/well_fov combinations: 4189


In [5]:
# Build expected .duckdb paths with vectorized string ops
df["file_path"] = (
    profile_base_dir.as_posix()
    + "/data/"
    + df["patient"]
    + "/image_based_profiles/0.converted_profiles/"
    + df["well_fov"]
    + "/"
    + df["well_fov"]
    + ".duckdb"
)

# Scan each converted_profiles directory once — faster than per-row .exists()
existing_duckdbs: set[str] = set()
candidate_dir_df = df[["patient", "well_fov"]].drop_duplicates()

for patient, well_fov in tqdm(
    candidate_dir_df.itertuples(index=False, name=None),
    total=len(candidate_dir_df),
    desc="Scanning converted_profiles directories",
):
    profile_dir = (
        profile_base_dir
        / "data"
        / patient
        / "image_based_profiles"
        / "0.converted_profiles"
        / well_fov
    )
    if profile_dir.exists():
        existing_duckdbs.update(
            p.as_posix() for p in profile_dir.glob("*.duckdb") if p.is_file()
        )

df["duckdb_exists"] = df["file_path"].isin(existing_duckdbs)

Scanning converted_profiles directories:   0%|          | 0/4189 [00:00<?, ?it/s]

In [6]:
# check each well fov to search for the number of extracted feature parquet files

df["num_parquets"] = df.apply(count_parquets, axis=1)

In [7]:
total = len(df)
# present = df["exists"].sum()
present = df.loc[df["duckdb_exists"] & (df["num_parquets"] == 101)].shape[0]
print(f"Total files to check : {total}")
print(f"Present              : {present}")
print(f"Missing              : {total - present}")

Total files to check : 4189
Present              : 1367
Missing              : 2822


In [8]:
# Write missing combinations to load_file.txt
df_missing = df.loc[
    ~df["duckdb_exists"] & (df["num_parquets"] == 101), ["patient", "well_fov"]
].reset_index(drop=True)
df_missing.to_csv(load_file_path, sep="\t", index=False, header=False)
print(f"Wrote {len(df_missing)} missing combinations to: {load_file_path}")

Wrote 1 missing combinations to: /home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/4.processing_image_based_profiles/load_data/load_file.txt


In [9]:
# Summary by patient
summary_df = (
    df.groupby("patient")[["duckdb_exists"]]
    .agg(total=("duckdb_exists", "count"), present=("duckdb_exists", "sum"))
    .assign(missing=lambda x: x["total"] - x["present"])
)
summary_df.reset_index()

,patient,total,present,missing
0,NF0014_T1,104,101,3
1,NF0014_T2,350,350,0
2,NF0016_T1,169,121,48
3,NF0018_T6,160,160,0
4,NF0021_T1,348,348,0
5,NF0030_T1,207,204,3
6,NF0035_T1,349,348,1
7,NF0037_T1,420,420,0
8,NF0037_T1_CQ1,693,693,0
9,NF0040_T1,420,418,2


In [10]:
# send to df
df = pd.DataFrame(
    list(
        pathlib.Path(f"{profile_base_dir}/data/NF0014_T1/extracted_features/C4-2").glob(
            "*.parquet"
        )
    )
)
df.rename(columns={0: "file_path"}, inplace=True)
df["file_name"] = df["file_path"].apply(lambda p: p.stem)
df["Compatment"] = df["file_name"].apply(lambda s: s.split("_")[0])
df["Channel"] = df["file_name"].apply(lambda s: s.split("_")[1])
df["FeatureType"] = df["file_name"].apply(lambda s: s.split("_")[2])
df.head()

,file_path,file_name,Compatment,Channel,FeatureType
0,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,Organoid_NoChannel_AreaSizeShape_CPU_features,Organoid,NoChannel,AreaSizeShape
1,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,Nuclei_ER_Granularity_CPU_features,Nuclei,ER,Granularity
2,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,Cytoplasm_ER-Mito_Colocalization_CPU_features,Cytoplasm,ER-Mito,Colocalization
3,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,Cell_AGP_Texture_CPU_features,Cell,AGP,Texture
4,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,Cytoplasm_DNA-Mito_Colocalization_CPU_features,Cytoplasm,DNA-Mito,Colocalization


In [11]:
df["Compatment"].value_counts()

Compatment
Nuclei           24
Organoid         23
Cytoplasm        23
Cell             23
Nucleocentric     8
Name: count, dtype: int64

In [12]:
df["Channel"].value_counts()

Channel
ER           18
AGP          18
DNA          18
Mito         18
NoChannel     5
ER-Mito       4
DNA-Mito      4
Mito-AGP      4
ER-AGP        4
DNA-ER        4
DNA-AGP       4
Name: count, dtype: int64

In [13]:
df["FeatureType"].value_counts()

FeatureType
Colocalization    24
SAMMed3D          20
Granularity       16
Texture           16
Intensity         16
AreaSizeShape      4
CHAMMI75           4
Neighbors          1
Name: count, dtype: int64